# RelCheck — Full Pipeline (Google Colab)

**CS298 Spring 2026 | Siddhi Patil**

Run cells top to bottom. This notebook:
1. Installs dependencies
2. Loads BLIP-2 + OWL-ViT + Mistral
3. Runs the full RelCheck pipeline on your 12 pilot images
4. Saves results to `pilot_results.csv`

**Before running:** Make sure you are on a GPU runtime.
Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install -q spacy transformers torch torchvision pillow together
!python -m spacy download en_core_web_sm -q
print('All dependencies installed.')

In [ ]:
# ── Cell 2: Upload RelCheck source files ─────────────────────────────────
# Upload all 4 .py files from your CS298/relcheck/ folder:
#   triple_extractor.py
#   relation_verifier.py
#   corrector.py
#   relcheck_pipeline.py

from google.colab import files
print('Upload the 4 RelCheck .py files now:')
files.upload()

In [ ]:
# ── Cell 3: Upload your 12 pilot images ──────────────────────────────────
import os
from google.colab import files

print('Upload your 12 images from the images/ folder:')
files.upload()

# List what was uploaded
image_paths = sorted([
    f for f in os.listdir('.')
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
])
print(f'Found {len(image_paths)} images:', image_paths)

In [ ]:
# ── Cell 4: Set your Together.ai API key ──────────────────────────────────
# Get your free key at: https://api.together.xyz
# Free tier = 5M tokens/month, more than enough for this project.

import os
os.environ['TOGETHER_API_KEY'] = 'PASTE_YOUR_KEY_HERE'   # <-- replace this
print('API key set.')

In [ ]:
# ── Cell 5: Load BLIP-2 for caption generation ───────────────────────────
# This downloads ~15GB the first time — takes 5-10 min on Colab

import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

blip2_name = 'Salesforce/blip2-flan-t5-xl'
processor = Blip2Processor.from_pretrained(blip2_name)
model     = Blip2ForConditionalGeneration.from_pretrained(
    blip2_name, torch_dtype=torch.float16
).to(device)
model.eval()
print('BLIP-2 loaded.')

In [ ]:
# ── Cell 6: Run RelCheck on all 12 pilot images ──────────────────────────

from relcheck_pipeline import RelCheckPipeline

pipeline = RelCheckPipeline()   # reads TOGETHER_API_KEY from environment

results = pipeline.run_batch(
    image_paths=image_paths,
    blip2_processor=processor,
    blip2_model=model,
    output_csv='pilot_results.csv',
)

print(f'\nDone! Processed {len(results)} images.')
print(f'Total hallucinations found: {sum(r.n_hallucinated for r in results)}')
print(f'Total corrections made:     {sum(r.n_corrected for r in results)}')

In [ ]:
# ── Cell 7: Display results as a table ───────────────────────────────────

import pandas as pd

df = pd.read_csv('pilot_results.csv')
display(df[['image_path', 'n_triples', 'n_hallucinated', 'n_corrected',
            'edit_rate', 'original_caption', 'corrected_caption']])

In [ ]:
# ── Cell 8: Show before/after for hallucinated examples ──────────────────

from IPython.display import display as ipy_display
from PIL import Image
import matplotlib.pyplot as plt

hallucinated_results = [r for r in results if r.any_hallucination]

if not hallucinated_results:
    print('No hallucinations found in pilot set — try with more images!')
else:
    for r in hallucinated_results[:5]:   # show at most 5
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        img = Image.open(r.image_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'{r.image_path}', fontsize=9)
        plt.tight_layout()
        plt.show()

        print(f'  ORIGINAL:  {r.original_caption}')
        print(f'  CORRECTED: {r.corrected_caption}')
        print(f'  Triples:   {[str(t) for t in r.triples]}')
        print()

In [ ]:
# ── Cell 9: Download results ──────────────────────────────────────────────

from google.colab import files
files.download('pilot_results.csv')
print('pilot_results.csv downloaded.')

## Smoke Test — Triple Extractor Only (no GPU needed)

Run this cell standalone to verify the Triple Extractor works, **before** loading BLIP-2.

In [ ]:
# ── Quick smoke test (run this first to sanity-check Stage 1) ────────────

from triple_extractor import TripleExtractor

extractor = TripleExtractor()

test_captions = [
    'A woman is holding a dog on a park bench near some trees.',
    'A cup is on the table next to a white mug.',
    'A man riding a bicycle near a red car on the street.',
    'Two birds are perched on a branch above a lake.',
    'The large brown dog is sitting beside a small child.',
]

for cap in test_captions:
    print(f'Caption: "{cap}"')
    triples = extractor.extract(cap)
    for t in triples:
        print(f'  → {t}')
    print()